<div style="background-color:#0e2233;background-image:linear-gradient(135deg,#0b1a29,#123049);color:#ffffff;padding:26px 28px 24px;border-radius:12px;border:1px solid #2a4a66;">
<div style="color:#a0d2ff;font-size:0.78em;font-weight:600;letter-spacing:0.18em;">AI &amp; ROBOTICS · NHÁNH AI ENGINEER · NOTEBOOK THỰC HÀNH</div>
<div style="width:56px;height:4px;background-color:#E07A1F;margin:14px 0 12px;"></div>
<h1 style="color:#ffffff;font-size:2.3em;line-height:1.15;margin:0;border:0;">Chương 32 · Dự án Phần III</h1>
<p style="color:#cfe6ff;font-size:1.12em;line-height:1.5;margin:10px 0 0;">Dự án tổng kết deep learning: phân loại 27 000 ảnh vệ tinh Sentinel-2 (EuroSAT) vào 10 loại đất. Baseline màu, MLP, CNN học từ đầu, tăng cường dữ liệu, ResNet-18 đóng băng và tinh chỉnh, đo test một lần, phân tích lỗi và ngưỡng tin cậy.</p>
</div>

### Sau notebook này bạn sẽ

- Kiểm tra dữ liệu ảnh trước khi chia, chia train, val, test cố định theo từng lớp.
- Đặt một baseline mạnh bằng công cụ của Phần II trước khi dùng mạng sâu.
- So MLP với CNN, dùng tăng cường dữ liệu hợp với loại ảnh, đọc khoảng cách train và val.
- Chọn giữa đóng băng và tinh chỉnh một mạng đã huấn luyện sẵn bằng val.
- Đo test một lần, đọc ma trận nhầm lẫn, nhìn ảnh sai và dùng ngưỡng tin cậy.

<div>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Thời gian chạy:</b> khoảng 1 giờ trên CPU 16 nhân (phần lớn là huấn luyện ở trang 2, 3 và bài tập 32.2)</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Runtime:</b> CPU là đủ, không cần GPU</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Trang web của chương:</b> <a href="https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch32-du-an-phan-3/01-bai-toan-va-du-lieu/" style="color:#1F5F8B;">mở bài đọc</a></span>
</div>

### Mục lục

1. [Trang 1 · Bài toán và dữ liệu](#trang-1)
2. [Trang 2 · CNN học từ đầu](#trang-2)
3. [Trang 3 · Transfer learning](#trang-3)
4. [Trang 4 · Đo test và phân tích lỗi](#trang-4)
5. [Trang 5 · Bài tập](#trang-5)

### Cách chạy

Bấm vào một ô code rồi nhấn **Shift + Enter** để chạy ô đó và nhảy sang ô kế tiếp. Muốn chạy hết một lượt thì chọn menu **Runtime > Run all**. Các ô nên chạy theo thứ tự từ trên xuống, vì ô sau dùng biến của ô trước.

<a name="trang-1" id="trang-1"></a>

## Trang 1 · Bài toán và dữ liệu

Đọc 27 000 ảnh vào RAM, kiểm tra trùng lặp, chia tập, hai baseline.

*Đọc kèm trên web: [Trang 1 · Bài toán và dữ liệu](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch32-du-an-phan-3/01-bai-toan-va-du-lieu/)*

In [ ]:
import numpy as np                                          # mảng số
import torch                                                # PyTorch
from torch import nn                                        # các lớp mạng
from torchvision.datasets import EuroSAT                    # bộ dữ liệu ảnh vệ tinh
ds = EuroSAT("data", download=True)                         # tải 94 MB lần đầu
X = torch.from_numpy(np.stack([np.array(ds.loader(p)) for p, _ in ds.samples]))   # đọc cả 27 000 ảnh vào RAM
X = X.permute(0, 3, 1, 2).contiguous()                      # (N, 3, 64, 64) cho PyTorch
y = torch.tensor(ds.targets)                                # nhãn 0 tới 9
print(X.shape, X.dtype, round(X.numel() / 1e6), "triệu byte")   # cỡ dữ liệu
for i, ten in enumerate(ds.classes):                        # mỗi lớp
    print(i, ten, int((y == i).sum()))                      # số ảnh

Output mong đợi:

```text
torch.Size([27000, 3, 64, 64]) torch.uint8 332 triệu byte
0 AnnualCrop 3000
1 Forest 3000
2 HerbaceousVegetation 3000
3 Highway 2500
4 Industrial 2500
5 Pasture 2000
6 PermanentCrop 2500
7 Residential 3000
8 River 2500
9 SeaLake 3000
```

In [ ]:
import hashlib                                              # mã băm
bam = [hashlib.md5(x.numpy().tobytes()).hexdigest() for x in X]   # mã băm từng ảnh
print(len(bam), len(set(bam)))                              # bao nhiêu ảnh khác nhau

Output mong đợi:

```text
27000 27000
```

In [ ]:
from sklearn.model_selection import train_test_split        # chia tập
chi_so = np.arange(len(y))                                  # chỉ số mọi ảnh
i_tr, i_con = train_test_split(chi_so, test_size=0.3, stratify=y.numpy(), random_state=0)   # 70% train
i_va, i_te = train_test_split(i_con, test_size=0.5, stratify=y.numpy()[i_con], random_state=0)   # 15% val, 15% test
i_tr, i_va, i_te = torch.from_numpy(i_tr), torch.from_numpy(i_va), torch.from_numpy(i_te)   # thành tensor
print(len(i_tr), len(i_va), len(i_te))                      # cỡ ba phần
print(torch.bincount(y[i_va]).tolist())                     # số ảnh mỗi lớp trong val

Output mong đợi:

```text
18900 4050 4050
[450, 450, 450, 375, 375, 300, 375, 450, 375, 450]
```

In [ ]:
lop_dong = torch.bincount(y[i_tr]).argmax()                 # lớp đông nhất của train
print(ds.classes[lop_dong], round((y[i_va] == lop_dong).float().mean().item(), 4))   # baseline một

Output mong đợi:

```text
AnnualCrop 0.1111
```

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier   # gradient boosting của Chương 18
def dac_trung_mau(X):                                       # 54 số chỉ về màu cho mỗi ảnh
    f = X.float().div(255).flatten(2)                       # (N, 3, 4096)
    hist = torch.stack([torch.stack([torch.histc(k, bins=16, min=0, max=1) for k in anh]) for anh in f])   # histogram 16 khoảng mỗi kênh
    return torch.cat([f.mean(2), f.std(2), hist.flatten(1) / 4096], 1).numpy()   # trung bình, độ lệch chuẩn, histogram
F_mau = dac_trung_mau(X)                                    # cho cả 27 000 ảnh
gb = HistGradientBoostingClassifier(random_state=0).fit(F_mau[i_tr], y[i_tr])   # chỉ học trên train
print(F_mau.shape, round(gb.score(F_mau[i_va], y[i_va]), 4))   # baseline hai

Output mong đợi:

```text
(27000, 54) 0.836
```

<a name="trang-2" id="trang-2"></a>

## Trang 2 · CNN học từ đầu

MLP và CNN học từ đầu, có và không có tăng cường dữ liệu (khoảng 30 phút trên CPU).

*Đọc kèm trên web: [Trang 2 · CNN học từ đầu](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch32-du-an-phan-3/02-cnn-tu-dau/)*

In [ ]:
import time                                                 # đo thời gian
tb = X[i_tr].float().div(255).mean((0, 2, 3)).view(1, 3, 1, 1)   # trung bình từng kênh, chỉ train
dl = X[i_tr].float().div(255).std((0, 2, 3)).view(1, 3, 1, 1)    # độ lệch chuẩn từng kênh, chỉ train
def chuan(xb):                                              # uint8 thành số thực đã chuẩn hóa
    return (xb.float().div(255) - tb) / dl                  # trung bình 0, độ lệch chuẩn 1
print([round(v, 4) for v in tb.flatten().tolist()], [round(v, 4) for v in dl.flatten().tolist()])   # thống kê

Output mong đợi:

```text
[0.3453, 0.381, 0.4082] [0.2036, 0.1376, 0.1161]
```

In [ ]:
ce = nn.CrossEntropyLoss()                                  # loss phân loại
@torch.no_grad()                                            # không cần gradient
def du_doan(m, idx, pre=chuan, bs=500):                     # logit cho các ảnh idx
    m.eval()                                                # chế độ đánh giá
    return torch.cat([m(pre(X[idx[i:i + bs]])) for i in range(0, len(idx), bs)])   # từng lô 500 ảnh
def do_acc(m, idx, pre=chuan):                              # độ chính xác
    return round((du_doan(m, idx, pre).argmax(1) == y[idx]).float().mean().item(), 4)   # tỷ lệ đoán đúng
def huan_luyen(m, so_epoch, lr=1e-3, tang=None, pre=chuan, tham_so=None, bs=128):   # AdamW và OneCycleLR
    torch.manual_seed(1)                                    # thứ tự lô cố định
    opt = torch.optim.AdamW(tham_so if tham_so is not None else m.parameters(), lr=lr, weight_decay=1e-4, fused=True)   # bản gộp, lặp lại được từng chữ số
    lich = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=so_epoch * ((len(i_tr) + bs - 1) // bs))   # warmup rồi cosine
    for ep in range(so_epoch):                              # mỗi epoch
        m.train()                                           # chế độ huấn luyện
        thu_tu, tong = i_tr[torch.randperm(len(i_tr))], 0.0   # xáo train
        for i in range(0, len(thu_tu), bs):                 # mỗi lô
            xb, yb = X[thu_tu[i:i + bs]], y[thu_tu[i:i + bs]]   # ảnh và nhãn của lô
            if tang is not None:                            # có tăng cường
                xb = tang(xb)                               # chỉ lúc huấn luyện
            loss = ce(m(pre(xb)), yb)                       # lượt xuôi
            opt.zero_grad()                                 # xóa gradient
            loss.backward()                                 # lượt ngược
            opt.step()                                      # cập nhật
            lich.step()                                     # OneCycleLR đếm theo lô
            tong += loss.item() * len(xb)                   # cộng loss
        print(ep + 1, round(tong / len(i_tr), 4), do_acc(m, i_va, pre))   # epoch, loss train, độ chính xác val
torch.manual_seed(0)                                        # cùng khởi tạo
mlp = nn.Sequential(nn.Flatten(), nn.Linear(3 * 64 * 64, 512), nn.ReLU(),   # duỗi ảnh, lớp ẩn 512
                    nn.Linear(512, 256), nn.ReLU(), nn.Linear(256, 10))     # lớp ẩn 256, 10 lớp ra
print(sum(p.numel() for p in mlp.parameters()))             # số tham số
huan_luyen(mlp, 15)                                         # 15 epoch

Output mong đợi:

```text
6425866
1 1.6186 0.5143
2 1.2844 0.5193
3 1.2611 0.5556
4 1.1679 0.5494
5 1.069 0.5795
6 0.9668 0.5667
7 0.821 0.6062
8 0.7278 0.5968
9 0.6421 0.618
10 0.5397 0.6284
11 0.4466 0.6341
12 0.3706 0.6472
13 0.3141 0.6496
14 0.2782 0.6496
15 0.2619 0.6509
```

In [ ]:
def khoi(vao, ra):                                          # tích chập, batch norm, ReLU, pooling
    return nn.Sequential(nn.Conv2d(vao, ra, 3, padding=1), nn.BatchNorm2d(ra), nn.ReLU(), nn.MaxPool2d(2))   # cạnh ảnh giảm một nửa
def tao_cnn():                                              # CNN bốn khối
    torch.manual_seed(0)                                    # cùng khởi tạo
    return nn.Sequential(khoi(3, 32), khoi(32, 64), khoi(64, 128), khoi(128, 256),   # 64, 32, 16, 8, 4
                         nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.2), nn.Linear(256, 10))   # trung bình toàn cục, lớp cuối
cnn = tao_cnn()                                             # mạng mới
print(sum(p.numel() for p in cnn.parameters()))             # số tham số
huan_luyen(cnn, 15)                                         # khoảng 14 phút trên CPU

Output mong đợi:

```text
391946
1 1.2943 0.7294
2 0.6912 0.8025
3 0.5424 0.8301
4 0.4767 0.8306
5 0.3822 0.8267
6 0.3155 0.9207
7 0.2783 0.8677
8 0.2379 0.8501
9 0.2044 0.9212
10 0.1744 0.9407
11 0.1557 0.9356
12 0.1373 0.9548
13 0.1148 0.962
14 0.1054 0.9635
15 0.0943 0.9635
```

In [ ]:
def lat_xoay(x):                                            # 8 phép lật và xoay ngẫu nhiên
    x = x.clone()                                           # không sửa dữ liệu gốc
    m = torch.rand(len(x)) < 0.5                            # chọn một nửa lô
    x[m] = x[m].flip(3)                                     # lật ngang
    m = torch.rand(len(x)) < 0.5                            # chọn một nửa lô
    x[m] = x[m].flip(2)                                     # lật dọc
    m = torch.rand(len(x)) < 0.5                            # chọn một nửa lô
    x[m] = x[m].transpose(2, 3)                             # lật theo đường chéo
    return x                                                # lô đã biến đổi
cnn_tang = tao_cnn()                                        # cùng khởi tạo
huan_luyen(cnn_tang, 15, tang=lat_xoay)                     # có tăng cường

Output mong đợi:

```text
1 1.2986 0.7257
2 0.7074 0.779
3 0.5725 0.8165
4 0.481 0.8543
5 0.4068 0.8758
6 0.3522 0.8926
7 0.3112 0.8746
8 0.2791 0.92
9 0.2418 0.9242
10 0.2154 0.9375
11 0.1948 0.9486
12 0.1731 0.9449
13 0.1552 0.9583
14 0.1421 0.9612
15 0.136 0.96
```

In [ ]:
for ten, m in [("không tăng cường", cnn), ("lật và xoay", cnn_tang)]:   # hai CNN
    print(ten, do_acc(m, i_tr[:4050]), do_acc(m, i_va))     # train so với val

Output mong đợi:

```text
không tăng cường 0.9795 0.9635
lật và xoay 0.9602 0.96
```

<div style="border-left:6px solid #7A6A58;background-color:#F4F1EC;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#5C4E3F;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">THỬ SỬA</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>1.</b> Huấn luyện cnn_tang 30 epoch thay vì 15. Khoảng cách với bản không tăng cường thay đổi thế nào?</p>
</div>

<a name="trang-3" id="trang-3"></a>

## Trang 3 · Transfer learning

ResNet-18 đóng băng và tinh chỉnh (khoảng 12 phút trên CPU).

*Đọc kèm trên web: [Trang 3 · Transfer learning](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch32-du-an-phan-3/03-transfer-learning/)*

In [ ]:
import torch.nn.functional as F                             # nội suy ảnh
from torchvision import models                              # mạng đã huấn luyện sẵn
from sklearn.linear_model import LogisticRegression         # hồi quy logistic
w = models.ResNet18_Weights.IMAGENET1K_V1                   # trọng số ImageNet
tb_in = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)   # trung bình ImageNet
dl_in = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)   # độ lệch chuẩn ImageNet
def chuan_in(kich):                                         # chuẩn hóa theo ImageNet, phóng ảnh nếu cần
    def f(xb):                                              # hàm tiền xử lý
        x = (xb.float().div(255) - tb_in) / dl_in           # chuẩn hóa
        return x if kich == 64 else F.interpolate(x, size=kich, mode="bilinear", align_corners=False)   # phóng ảnh
    return f                                                # trả về hàm
for kich in (64, 128):                                      # ảnh gốc và ảnh phóng gấp đôi
    torch.manual_seed(0)                                    # cùng khởi tạo
    than = models.resnet18(weights=w)                       # ResNet-18 ImageNet
    than.fc = nn.Identity()                                 # giữ vector 512 số
    F_tr, F_va = du_doan(than, i_tr, chuan_in(kich)).numpy(), du_doan(than, i_va, chuan_in(kich)).numpy()   # đặc trưng
    lr_dau = LogisticRegression(max_iter=3000).fit(F_tr, y[i_tr])   # chỉ học lớp cuối
    print(kich, F_tr.shape, round(lr_dau.score(F_va, y[i_va]), 4))   # độ chính xác val

Output mong đợi:

```text
64 (18900, 512) 0.9044
128 (18900, 512) 0.9264
```

In [ ]:
torch.manual_seed(0)                                        # cùng khởi tạo
net = models.resnet18(weights=w)                            # ResNet-18 ImageNet
net.fc = nn.Linear(512, 10)                                 # lớp cuối mới cho 10 loại đất
huan_luyen(net, 5, tang=lat_xoay, pre=chuan_in(64))         # tinh chỉnh toàn bộ, khoảng 9 phút trên CPU

Output mong đợi:

```text
1 0.5214 0.9323
2 0.2233 0.9101
3 0.1463 0.9642
4 0.0847 0.9741
5 0.0551 0.9807
```

<a name="trang-4" id="trang-4"></a>

## Trang 4 · Đo test và phân tích lỗi

Đo test một lần, ma trận nhầm lẫn, ngưỡng tin cậy.

*Đọc kèm trên web: [Trang 4 · Đo test và phân tích lỗi](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch32-du-an-phan-3/04-do-test-va-phan-tich-loi/)*

In [ ]:
P_te = du_doan(net, i_te, chuan_in(64)).softmax(1)          # xác suất cho mỗi ảnh test
doan = P_te.argmax(1)                                       # lớp đoán
dung = doan == y[i_te]                                      # đúng hay sai
print(round(dung.float().mean().item(), 4), int((~dung).sum()), len(i_te))   # độ chính xác, số ảnh sai

Output mong đợi:

```text
0.9753 100 4050
```

In [ ]:
from sklearn.metrics import confusion_matrix                # ma trận nhầm lẫn
M = confusion_matrix(y[i_te], doan)                         # hàng thật, cột đoán
for i, ten in enumerate(ds.classes):                        # mỗi lớp
    print(f"{ten:21s} {M[i, i] / M[i].sum():.4f}")        # tỷ lệ đúng của lớp
cap = sorted(((int(M[i, j]), ds.classes[i], ds.classes[j]) for i in range(10) for j in range(10) if i != j), reverse=True)   # các cặp nhầm
print(cap[:4])                                              # bốn cặp nhầm nhiều nhất

Output mong đợi:

```text
AnnualCrop            0.9756
Forest                0.9911
HerbaceousVegetation  0.9756
Highway               0.9787
Industrial            0.9867
Pasture               0.9400
PermanentCrop         0.9280
Residential           0.9956
River                 0.9707
SeaLake               0.9933
[(14, 'PermanentCrop', 'HerbaceousVegetation'), (7, 'PermanentCrop', 'AnnualCrop'), (7, 'Pasture', 'HerbaceousVegetation'), (7, 'AnnualCrop', 'PermanentCrop')]
```

In [ ]:
tin = P_te.max(1).values                                    # xác suất của lớp được chọn
for nguong in (0.9, 0.99):                                  # hai ngưỡng
    giu = tin >= nguong                                     # ảnh mô hình tự quyết
    print(nguong, round(giu.float().mean().item(), 4), round(dung[giu].float().mean().item(), 4),   # tỷ lệ giữ, độ chính xác
          round(dung[~giu].float().mean().item(), 4))       # độ chính xác phần chuyển cho người

Output mong đợi:

```text
0.9 0.9412 0.9924 0.7017
0.99 0.8207 0.9991 0.8664
```

<a name="trang-5" id="trang-5"></a>

## Trang 5 · Bài tập

Ba bài tập. Đoán trước, rồi mới mở đáp án.

*Đọc kèm trên web: [Trang 5 · Bài tập](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch32-du-an-phan-3/05-bai-tap/)*

### Bài 32.1 · Tăng cường lúc dự đoán

Trung bình xác suất của 8 phép lật và xoay, cho cả hai CNN của trang 2.
<details><summary>Xem đáp án</summary>

Chỉ mô hình đã học bằng lật và xoay được lợi (96,0% lên 96,27%); mô hình kia giảm. Code ở ô ngay dưới.
</details>

In [ ]:
def du_doan_8(m, idx):                                      # trung bình 8 biến thể
    P = 0                                                   # tổng xác suất
    for a in (0, 1):                                        # lật ngang hay không
        for b in (0, 1):                                    # lật dọc hay không
            for c in (0, 1):                                # lật chéo hay không
                def bien(xb, a=a, b=b, c=c):                # một trong 8 phép
                    if a:                                   # lật ngang
                        xb = xb.flip(3)                     # đảo cột
                    if b:                                   # lật dọc
                        xb = xb.flip(2)                     # đảo hàng
                    if c:                                   # lật chéo
                        xb = xb.transpose(2, 3)             # đổi hàng thành cột
                    return chuan(xb)                        # chuẩn hóa
                P = P + du_doan(m, idx, bien).softmax(1)    # cộng xác suất
    return P / 8                                            # trung bình
for ten, m in [("không tăng cường", cnn), ("lật và xoay", cnn_tang)]:   # hai CNN
    print(ten, do_acc(m, i_va), round((du_doan_8(m, i_va).argmax(1) == y[i_va]).float().mean().item(), 4))   # một ảnh so với 8 biến thể

Output mong đợi:

```text
không tăng cường 0.9635 0.9605
lật và xoay 0.96 0.9627
```

### Bài 32.2 · Trọng số hay kiến trúc

ResNet-18 cùng cách huấn luyện nhưng weights=None.
<details><summary>Xem đáp án</summary>

90,1% sau 5 epoch so với 98,1% khi dùng trọng số ImageNet: với ngân sách ngắn, trọng số đã học đóng góp gần 8 điểm. Code ở ô ngay dưới.
</details>

In [ ]:
torch.manual_seed(0)                                        # cùng khởi tạo
net_moi = models.resnet18(weights=None)                     # trọng số ngẫu nhiên
net_moi.fc = nn.Linear(512, 10)                             # 10 loại đất
huan_luyen(net_moi, 5, tang=lat_xoay, pre=chuan_in(64))     # cùng 5 epoch, khoảng 9 phút trên CPU

Output mong đợi:

```text
1 1.0187 0.6817
2 0.7209 0.6183
3 0.5591 0.7121
4 0.3972 0.8773
5 0.2904 0.9012
```

### Bài 32.3 · Chọn ngưỡng theo yêu cầu

Ngưỡng thấp nhất để ảnh mô hình tự quyết đúng ít nhất 99,5%.
<details><summary>Xem đáp án</summary>

Ngưỡng 0,95: đúng 99,51%, chuyển 348 ảnh cho người. Code ở ô ngay dưới.
</details>

In [ ]:
for nguong in (0.8, 0.9, 0.95, 0.97):                       # bốn ngưỡng
    giu = tin >= nguong                                     # ảnh mô hình tự quyết
    print(nguong, int((~giu).sum()), round(dung[giu].float().mean().item(), 4))   # số ảnh chuyển cho người, độ chính xác

Output mong đợi:

```text
0.8 147 0.99
0.9 238 0.9924
0.95 348 0.9951
0.97 475 0.9978
```